In [1]:
!pip install transformers[torch] datasets scikit-learn pandas -q

In [2]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
MODEL_NAME = "distilbert-base-uncased"
CSV_FILE = "labeled_dataset.csv"

# Check for T4 GPU
if not torch.cuda.is_available():
    print("="*50)
    print("⚠️ WARNING: GPU not found. Training will be very slow.")
    print("   Go to Runtime -> Change runtime type -> Select 'T4 GPU'")
    print("="*50)
else:
    print("✅ GPU (T4) found! Proceeding with training.")

✅ GPU (T4) found! Proceeding with training.


In [4]:
if not os.path.exists(CSV_FILE):
    print(f"\n❌ ERROR: File '{CSV_FILE}' not found.")
    print("   Please upload your 'labeled_dataset.csv' file to the Colab environment.")
    # Stop the script if the file isn't there
    raise FileNotFoundError(f"{CSV_FILE} not found. Please upload it to Colab.")

print(f"\nLoading dataset from '{CSV_FILE}'...")
df = pd.read_csv(CSV_FILE)


Loading dataset from 'labeled_dataset.csv'...


In [5]:
print("Starting feature engineering...")

# Get all columns *except* the label column
feature_columns = [col for col in df.columns if col != 'label']

# Replace 'nan' (which pandas creates for empty cells) with an empty string
df[feature_columns] = df[feature_columns].fillna('')

# Create the 'text' column by joining all feature columns
# We explicitly convert to string and join with a space.
df['text'] = df[feature_columns].apply(
    lambda x: ' '.join(x.astype(str)),
    axis=1
)

print("  - 'text' feature created.")


Starting feature engineering...
  - 'text' feature created.


In [6]:
label_map = {'benign': 0, 'reverse_shell': 1, 'priv_esc': 2}

# Create the integer 'label' column
df['label_int'] = df['label'].map(label_map)

# Drop any rows where the label might be unknown (good practice)
df = df.dropna(subset=['label_int'])
df['label_int'] = df['label_int'].astype(int)

# The model needs these maps to understand what '0', '1', and '2' mean
id2label = {v: k for k, v in label_map.items()}
label2id = {k: v for k, v in label_map.items()}
NUM_LABELS = len(label_map)

print(f"  - Labels encoded. Found {NUM_LABELS} classes: {list(label_map.keys())}")

  - Labels encoded. Found 3 classes: ['benign', 'reverse_shell', 'priv_esc']


In [7]:
final_df = df[['text', 'label_int']].rename(columns={'label_int': 'label'})

print("\nDataset preprocessing complete. Example entry:")
print(final_df.iloc[0]['text'])
print(f"Label: {final_df.iloc[0]['label']} ({id2label[final_df.iloc[0]['label']]})")



Dataset preprocessing complete. Example entry:
exec_create: ps aux a04b2d08578be02135c9d66c98c9d4a03699a13b1caf0169f2e53ea2759345b9 paypal-auth-service container exec_create: ps aux {'ID': 'a04b2d08578be02135c9d66c98c9d4a03699a13b1caf0169f2e53ea2759345b9', 'Attributes': {'com.docker.compose.config-hash': 'fc76da42427d0334d13be59305aa6e99d15aa9e3e9a6cfd21510426287b0567b', 'com.docker.compose.container-number': '1', 'com.docker.compose.depends_on': '', 'com.docker.compose.image': 'sha256:b6bdf44cff31c7a541972a9aec98cf2e9e68dccc87adbf98cfc132bde827ba37', 'com.docker.compose.oneoff': 'False', 'com.docker.compose.project': 'paypal', 'com.docker.compose.project.config_files': '/Users/anuhya/Documents/udays_micro/LLM_Based_CyberThreatDetection/docker-compose.yml', 'com.docker.compose.project.working_dir': '/Users/anuhya/Documents/udays_micro/LLM_Based_CyberThreatDetection', 'com.docker.compose.service': 'auth-service', 'com.docker.compose.version': '2.29.7', 'desktop.docker.io/binds/0/Source

In [9]:
train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df['label'])

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"\nData split: {len(train_dataset)} training, {len(test_dataset)} testing.")

# --- 4.2. Load Tokenizer & Tokenize ---
print(f"Loading tokenizer for '{MODEL_NAME}'...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # 'truncation=True' cuts off logs that are too long
    # 'padding="max_length"' adds padding to logs that are too short
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256  # You can tune this length
    )

print("Tokenizing datasets (this may take a moment)...")
# .map() is a fast way to apply the function to the whole dataset
# Use num_proc to speed up tokenization using multiple CPU cores
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count())
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, num_proc=os.cpu_count())

# Remove the raw 'text' column, as it's no longer needed
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"])
tokenized_train_dataset.set_format("torch")
tokenized_test_dataset.set_format("torch")
print("Tokenization complete.")


Data split: 52817 training, 13205 testing.
Loading tokenizer for 'distilbert-base-uncased'...
Tokenizing datasets (this may take a moment)...


Map (num_proc=2):   0%|          | 0/52817 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/13205 [00:00<?, ? examples/s]

Tokenization complete.


In [10]:
print(f"Loading model '{MODEL_NAME}' for {NUM_LABELS}-class classification...")
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,  # Link IDs to string labels
    label2id=label2id   # Link string labels to IDs
)

Loading model 'distilbert-base-uncased' for 3-class classification...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics, using 'weighted' for our imbalanced classes
    f1 = f1_score(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [13]:
PER_DEVICE_BATCH_SIZE = 32 # Increased batch size for better GPU utilization

# Calculate steps per epoch for logging/saving
steps_per_epoch = len(tokenized_train_dataset) // PER_DEVICE_BATCH_SIZE
if steps_per_epoch == 0:
    steps_per_epoch = 1 # Avoid division by zero if dataset is smaller than batch

print(f"Batch Size: {PER_DEVICE_BATCH_SIZE}, Steps per Epoch: {steps_per_epoch}")

# These arguments control the entire training process
training_args = TrainingArguments(
    output_dir="./results",               # Where to save the model
    logging_dir="./logs",                 # Where to save logs
    num_train_epochs=3,                   # 3 epochs is a good starting point
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE, # Batch size for T4 GPU
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    warmup_steps=500,                     # Steps to "warm up" the learning rate
    weight_decay=0.01,                    # Strength of regularization
    eval_strategy="epoch",          # Evaluate every epoch
    logging_steps=steps_per_epoch,   # Log every epoch
    save_strategy="epoch",      # Save every epoch
    # ------------------------------------------
    fp16=True,                            # Enable mixed precision training for speed
    dataloader_num_workers=2,             # Number of workers for data loading

    load_best_model_at_end=True,          # Automatically load the best model
    metric_for_best_model="f1",           # Use F1-score to find the best model
    report_to="none"                      # Disable reporting to WandB/etc.
)

Batch Size: 32, Steps per Epoch: 1650


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Pass our metrics function
)


/tmp/ipython-input-2073410088.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
print("\n" + "="*50)
print("🚀 STARTING MODEL TRAINING...")
print("="*50)
trainer.train()

print("\n🎉 Training complete!")


🚀 STARTING MODEL TRAINING...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.106700,0.059684,0.969178,0.969257,0.973627,0.969178
2,0.062000,0.059479,0.969254,0.969338,0.973703,0.969254
3,0.058800,0.059320,0.969254,0.969338,0.973703,0.969254



🎉 Training complete!


In [17]:
print("\n" + "="*50)
print("📊 EVALUATING FINAL MODEL ON TEST SET...")
print("="*50)
eval_results = trainer.evaluate()
print(eval_results)

# Save the final, best model
final_model_path = "./final_model"
trainer.save_model(final_model_path)
print(f"\n✅ Final model saved to '{final_model_path}'")





📊 EVALUATING FINAL MODEL ON TEST SET...


{'eval_loss': 0.059479255229234695, 'eval_accuracy': 0.9692540704278683, 'eval_f1': 0.9693380227426127, 'eval_precision': 0.9737026974436507, 'eval_recall': 0.9692540704278683, 'eval_runtime': 20.6167, 'eval_samples_per_second': 640.5, 'eval_steps_per_second': 20.032, 'epoch': 3.0}

✅ Final model saved to './final_model'


In [18]:
from transformers import pipeline

print("\n" + "="*50)
print("🧪 RUNNING INFERENCE EXAMPLE...")
print("="*50)

# Load our saved model into a pipeline for easy use
# device=0 tells it to use the GPU
pipe = pipeline(
    "text-classification",
    model=final_model_path,
    tokenizer=tokenizer,
    device=0
)

# Grab a real example from our test set
test_example_text = test_df.iloc[0]['text']
test_example_label = id2label[test_df.iloc[0]['label']]

print(f"Test Log:\n{test_example_text}\n")
print(f"True Label: {test_example_label}\n")

# Run the prediction
# Explicitly set max_length and truncation for inference
prediction = pipe(test_example_text, max_length=256, truncation=True)
print(f"Model Prediction:\n{prediction}")

print("\nInference example complete. Script finished.")

Device set to use cuda:0



🧪 RUNNING INFERENCE EXAMPLE...
Test Log:
   volume destroy {'ID': '2930bd70b3d9d11aee65e6bfaa98767c432a319dd19372d312799e653eb8d525', 'Attributes': {'driver': 'local'}} local 1762872957 1762872957043610970

True Label: priv_esc

Model Prediction:
[{'label': 'priv_esc', 'score': 0.999931812286377}]

Inference example complete. Script finished.


In [19]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("\n" + "="*50)
print("📊 FULL TEST SET EVALUATION")
print("="*50)

# Get predictions on the test set
# The trainer's predict method returns an object with predictions, label_ids, and metrics
predictions_output = trainer.predict(tokenized_test_dataset)

# Extract true labels and predicted labels
true_labels = predictions_output.label_ids
predicted_labels = np.argmax(predictions_output.predictions, axis=-1)

# Calculate and display the confusion matrix
print("\nConfusion Matrix:")
# Use the actual label names for clarity, mapping from integer IDs
cm = confusion_matrix(true_labels, predicted_labels)
# Print with row/column labels if possible, or just the matrix
print(cm) # Simple print of the matrix

# Calculate and display the classification report (precision, recall, f1-score per class)
print("\nClassification Report:")
# The target_names argument makes the report much more readable by using the actual class names
report = classification_report(true_labels, predicted_labels, target_names=list(label_map.keys()))
print(report)

# The overall metrics are also available in predictions_output.metrics
print("\nOverall Metrics from Trainer.predict():")
print(predictions_output.metrics)

print("\nFull test set evaluation complete.")


📊 FULL TEST SET EVALUATION



Confusion Matrix:
[[2596  406    0]
 [   0 2400    0]
 [   0    0 7803]]

Classification Report:
               precision    recall  f1-score   support

       benign       1.00      0.86      0.93      3002
reverse_shell       0.86      1.00      0.92      2400
     priv_esc       1.00      1.00      1.00      7803

     accuracy                           0.97     13205
    macro avg       0.95      0.95      0.95     13205
 weighted avg       0.97      0.97      0.97     13205


Overall Metrics from Trainer.predict():
{'test_loss': 0.059479255229234695, 'test_accuracy': 0.9692540704278683, 'test_f1': 0.9693380227426127, 'test_precision': 0.9737026974436507, 'test_recall': 0.9692540704278683, 'test_runtime': 20.788, 'test_samples_per_second': 635.224, 'test_steps_per_second': 19.867}

Full test set evaluation complete.


In [20]:
import os
from google.colab import files

print("\n" + "="*50)
print("📦 ZIPPING AND PREPARING MODEL FOR DOWNLOAD...")
print("="*50)

# Define the path to the model directory and the desired zip file name
model_dir = "./final_model"
zip_file_name = "final_model.zip"
zip_file_path = os.path.join("/content", zip_file_name) # Save to /content for easy access

# Check if the model directory exists
if not os.path.exists(model_dir):
    print(f"❌ Error: Model directory '{model_dir}' not found.")
else:
    # Use a shell command to zip the directory
    # -r: recursive (include subdirectories)
    # -q: quiet (suppress output)
    zip_command = f"zip -r -q {zip_file_path} {model_dir}"
    print(f"Running zip command: {zip_command}")
    os.system(zip_command)

    # Check if the zip file was created
    if os.path.exists(zip_file_path):
        print(f"✅ Successfully created zip file: '{zip_file_name}'")
        print("Initiating download...")
        # Provide the file for download
        files.download(zip_file_path)
        print("\nDownload initiated. Look for the file in your browser's downloads.")
    else:
        print(f"❌ Error: Zip file '{zip_file_name}' was not created.")

print("\nZipping and download preparation complete.")


📦 ZIPPING AND PREPARING MODEL FOR DOWNLOAD...
Running zip command: zip -r -q /content/final_model.zip ./final_model
✅ Successfully created zip file: 'final_model.zip'
Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download initiated. Look for the file in your browser's downloads.

Zipping and download preparation complete.
